# 07 - Modelo final

En este notebook reconstruimos el modelo final seleccionado en el proyecto.

El mejor resultado global se obtuvo con **Naive Bayes** usando el conjunto combinado de atributos originales y metricas relacionales. En concreto, se utiliza la configuracion base:

- `CategoricalNB(alpha=1)`
- metricas continuas discretizadas con `n_bins=5`
- atributos originales de palabras + metricas relacionales + comunidad Louvain

Al final se muestran los resultados principales del modelo seleccionado para dejar claro cual es el modelo final del proyecto.

## 1. Importacion de librerias

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Bins whose width are too small.*", category=UserWarning)

## 2. Carga de datos procesados

In [ ]:
cwd = Path.cwd()
ROOT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = ROOT_DIR / "data" / "processed"
contenido = pd.read_csv(DATA_DIR / "cora_content_procesado.csv")
metricas = pd.read_csv(DATA_DIR / "cora_metricas_relacionales.csv")

print("Contenido:", contenido.shape)
print("Metricas:", metricas.shape)

## 3. Construccion del conjunto combinado

In [ ]:
columnas_palabras = [col for col in contenido.columns if col.startswith("palabra_")]

metricas_continuas = [
    "degree",
    "degree_centrality",
    "betweenness_centrality",
    "closeness_centrality",
    "clustering_coefficient",
    "pagerank"
]

metrica_categorica = ["louvain_community"]
columnas_metricas = metricas_continuas + metrica_categorica

datos_modelo = contenido.merge(
    metricas[["articulo_id"] + columnas_metricas],
    on="articulo_id",
    how="inner"
)

X = datos_modelo[columnas_palabras + columnas_metricas]
y = datos_modelo["clase"]

print("X combinado:", X.shape)
print("y:", y.shape)

## 4. Particion train/test

Se utiliza la misma particion estratificada que en los notebooks de modelos para mantener una comparacion coherente.

In [ ]:
indices_train, indices_test = train_test_split(
    X.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X.loc[indices_train]
X_test = X.loc[indices_test]
y_train = y.loc[indices_train]
y_test = y.loc[indices_test]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 5. Pipeline de preprocesado para Naive Bayes

`CategoricalNB` necesita atributos discretos. Por eso, las metricas continuas se discretizan en 5 intervalos dentro del propio `Pipeline`, mientras que las palabras binarias y la comunidad Louvain se mantienen como variables categoricas.

In [ ]:
N_BINS_FINAL = 5

preprocesador_final = ColumnTransformer(
    transformers=[
        (
            "metricas_continuas",
            KBinsDiscretizer(
                n_bins=N_BINS_FINAL,
                encode="ordinal",
                strategy="quantile"
            ),
            metricas_continuas
        ),
        (
            "categoricas_directas",
            "passthrough",
            columnas_palabras + metrica_categorica
        )
    ],
    remainder="drop"
)

print("Metricas continuas discretizadas:", metricas_continuas)
print("Atributos categoricos directos:", len(columnas_palabras) + len(metrica_categorica))

## 6. Entrenamiento del modelo final

In [ ]:
ALPHA_FINAL = 1
num_comunidades = int(metricas["louvain_community"].max()) + 1

# El orden debe coincidir con el ColumnTransformer: primero metricas continuas y despues categoricas directas.
min_categories_final = (
    [N_BINS_FINAL] * len(metricas_continuas)
    + [2] * len(columnas_palabras)
    + [num_comunidades]
)

modelo_final = Pipeline(
    steps=[
        ("preprocesado", preprocesador_final),
        (
            "naive_bayes",
            CategoricalNB(
                alpha=ALPHA_FINAL,
                min_categories=min_categories_final
            )
        )
    ]
)

modelo_final.fit(X_train, y_train)

y_pred = modelo_final.predict(X_test)
accuracy_final = accuracy_score(y_test, y_pred)

print("Modelo final: Naive Bayes combinado base")
print("alpha:", ALPHA_FINAL)
print("n_bins:", N_BINS_FINAL)
print("Accuracy en test:", accuracy_final)

## 7. Evaluacion final

In [ ]:
print("Matriz de confusion:")
print(confusion_matrix(y_test, y_pred))

print("\nInforme de clasificacion:")
print(classification_report(y_test, y_pred))

## 8. Resumen del modelo seleccionado

Mostramos de forma resumida el modelo elegido, sus parametros principales y el rendimiento obtenido en el conjunto de prueba.

In [ ]:
print("Modelo seleccionado: Naive Bayes combinado base")
print("Atributos utilizados: palabras originales + metricas relacionales")
print("Clasificador: CategoricalNB")
print("alpha:", ALPHA_FINAL)
print("n_bins para metricas continuas:", N_BINS_FINAL)
print("Accuracy final en test:", accuracy_final)

## 9. Conclusion

El modelo final seleccionado es **Naive Bayes con atributos originales y metricas relacionales combinadas**. Esta configuracion base obtiene el mejor rendimiento global del proyecto y por eso queda identificada como la solucion final.